## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample
from statistics import mean

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
from scipy.stats import linregress
from scipy.optimize import curve_fit
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = ["test_noise_floor_26.04.24"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

### Loading

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

### Signal Processing

In [ ]:
noise_range = 40
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    signals_df = signals_df.astype("int32")
    signals_np = signals_df.to_numpy()
    print(signals_np.shape)
    baselines = signals_np[:, :noise_range].mean(axis=1).reshape(-1, 1)
    signals_np = -signals_np + baselines
    signals_df.columns = signals_df.columns.map(int)
    signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)
    exp_data["signals_df"] = signals_df

### Histograms

In [ ]:
e_bin_width = 1
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    histo_data = signals_df.iloc[:, :noise_range]
    histo_times = np.tile(
        histo_data.columns.map(lambda x: x * 2),
        histo_data.shape[0]
    )
    histo_es = np.ravel(histo_data.to_numpy(), order="F")
    # print(histo_es.min())
    # print(histo_es.max())
    
    # time_bins = np.arange(0, 82, 2)
    e_min = -500
    e_max = 750
    e_bins = np.linspace(e_min, e_max, int((e_max - e_min) / e_bin_width)+1)
    # print(e_bins)
    # histo_results = np.histogram2d(histo_times, histo_es, bins=(time_bins, e_bins))
    histo_results = np.histogram(histo_es, bins=e_bins)
    # print(histo_results)
    histo_counts, histo_e_bins = histo_results
    # histo_counts, histo_time_bins, histo_e_bins = histo_results
    # collapsed_histo_energy = histo_counts.mean(axis=0)
    exp_data["histo_results"] = {
        "counts": histo_counts,
        # "time_bins": histo_time_bins,
        "e_bins": histo_e_bins,
        # "collapsed_histo_energy": collapsed_histo_energy
    }

In [ ]:
e_bin_width_2d = 10
time_bin_width_2d = 2

for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    histo_data = signals_df.iloc[:, :noise_range]
    histo_times = np.tile(
        histo_data.columns.map(lambda x: x * 2),
        histo_data.shape[0]
    )
    histo_es = np.ravel(histo_data.to_numpy(), order="F")

    time_bins = np.arange(0, 82, 2)
    e_min = -500
    e_max = 750
    e_bins = np.linspace(e_min, e_max, int((e_max - e_min) / e_bin_width)+1)
    # print(e_bins)
    histo_results = np.histogram2d(histo_times, histo_es, bins=(time_bins, e_bins))
    histo_counts, histo_time_bins, histo_e_bins = histo_results
    exp_data["histo_2d_results"] = {
        "counts": histo_counts,
        "time_bins": histo_time_bins,
        "e_bins": histo_e_bins
    }

### Gaussian Fit

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    histo_results = exp_data["histo_results"]
    collapsed_histo = histo_results["counts"]
    e_bins = histo_results["e_bins"]
    e_mids = (e_bins[1:] + e_bins[:-1]) / 2

    fit_params, _ = curve_fit(proc.gaussian, e_mids, collapsed_histo)
    print(fit_params)
    histo_results["fit_params"] = fit_params

### Pulse Selection

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

In [ ]:
bg_blue_map_codes = [
    [255, 255, 255],
    [76, 148, 255]
]
map_colors = [[value/255 for value in color] for color in bg_blue_map_codes]
bg_blue_map = mpl.colors.LinearSegmentedColormap.from_list("vaporwave", map_colors, N=256)
# vaporwave = vaporwave.resampled(256)
bg_blue_map

In [ ]:
e_conversion_value = 17/100
threshold = 250
e_bin_width_2d = 10
time_bin_width_2d = 2
e_min = -500
e_max = 750

for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    histo_results = exp_data["histo_results"]
    e_bins = histo_results["e_bins"]
    collapsed_histo_energy = histo_results["counts"]
    fit_params = histo_results["fit_params"]
    mu, sigma, peak_height = fit_params
    # fit_params = histo_results["fit_params"]

    histo_data = signals_df.iloc[:, :noise_range]
    histo_times = np.tile(
        histo_data.columns.map(lambda x: x * 2),
        histo_data.shape[0]
    )
    histo_es = np.ravel(histo_data.to_numpy(), order="F")

    time_bins_2d = np.arange(0, 82, 2)
    e_bins_2d = np.linspace(e_min, e_max, int((e_max - e_min) / e_bin_width_2d)+1)

    e_mids = (e_bins[1:] + e_bins[:-1]) / 2
    below_threshold = e_mids < threshold
    above_threshold = e_mids >= threshold
    gauss_es = np.linspace(-50, 50, 1000)
    gauss_counts = proc.gaussian(gauss_es, *fit_params)
    
    fig, ax = plt.subplots(figsize=(14, 12))
    divider = make_axes_locatable(ax)
    ax_h = divider.append_axes("right", 8, pad=0.1, sharey=ax)

    pulse_x = signals_df.columns.map(lambda x: int(x) * 2)
    pulse_x_subset = pulse_x[:noise_range]
    signals_np = signals_df.to_numpy()
    # for pulse_y_subset in signals_np[:, :noise_range]:
        # ax.plot(pulse_x_subset, pulse_y_subset, color=bg_blue, lw=2, alpha=0.01)
        # ax.plot(pulse_x_subset, pulse_y_subset, "o", color=bg_blue, ms=5, alpha=0.01)
    ax.hist2d(histo_times, histo_es, bins=(time_bins, e_bins), cmap=bg_blue_map)
    ax.set_xlim(0, 80)
    ax.set_ylim(-15, 15)
    ax.set_xlabel("Time (ns)", fontsize=fontsize)
    ax.set_ylabel("Pulse height (ADC channel)", fontsize=fontsize)
    ax.xaxis.set_major_locator(mpl.ticker.LinearLocator(numticks=3))
    ax.tick_params(labelsize=fontsize)
    # ax.axhline(threshold, color="red", linestyle="dashed")

    # ax_h.plot(collapsed_histo_energy, e_mids)
    # ax_h.fill_betweenx(e_mids, 0, collapsed_histo_energy, where=below_threshold, color="grey", interpolate=True)
    ax_h.fill_betweenx(e_mids, 0, collapsed_histo_energy, color=bg_bluegrey)
    # ax_h.axhline(threshold, color="red", linestyle="dashed")
    ax_h.plot(gauss_counts, gauss_es, "-", color=bg_blue, lw=3)
    ax_h.hlines([mu, mu-sigma], 0, 1.2e6, color="black", lw=3)
    ax_h.annotate(r"$\mu$", (1.15e6, mu), ha="right", va="bottom", fontsize=fontsize)
    ax_h.annotate(r"$\sigma$", (1.15e6, mu-sigma), ha="right", va="bottom", fontsize=fontsize)

    # ax_h.set_xscale("log")
    ax_h.tick_params(labelsize=fontsize)
    ax_h.set_xlabel(r"Counts (x $10^6$)", fontsize=fontsize)
    ax_h.yaxis.set_tick_params(labelleft=False)
    ax_h.set_xlim(0, 1.2e6)
    ax_h.xaxis.set_major_formatter(lambda x, pos: "" if pos == 0 else f"{x / 1e6:.1f}")
    
    # colorbar = fig.colorbar(histo_img, ax=ax_h, fraction=0.1)
    # colorbar.ax.tick_params(labelsize=fontsize)
    # colorbar.ax.set_ylabel("Counts", fontsize=fontsize)
    # ax_h.set_xlabel("Time (ns)", fontsize=fontsize)
    # ax_h.set_ylabel("Pulse height (keVee)", fontsize=fontsize)
    # ax_h.tick_params(labelsize=fontsize)
    # ax_h.set_ylim(-2.5, 2.5)
    # h_counts, h_bins = np.histogram(histo_data, bins="auto")
    # h_bins_mids = (h_bins[:-1] + h_bins[1:]) / 2
    # ax_h.bar(h_bins_mids, h_counts)